Este notebook estará orientado a exploración de # las 5 tablas: perfil, nulos, duplicados, claves, registro de incidencias.

Trabajo sobre la copia creada en el diccionario

In [ ]:
import sys; sys.path.append("..")
from src.carga import cargar_bronze

dfs = cargar_bronze()
dfs.keys()   # agencias, empresas, ocupaciones, reservaciones, segmentos_mercados

In [ ]:
ag = dfs["agencias"]

In [ ]:
# Con herramientas de pandas, valido información y nulos simples
ag.info()

In [ ]:
# ¿Cuántos valores distintos existen en la columna? / Ver cardinalidad
ag["ID_Agencia"].nunique()

In [ ]:
# ¿Todos los valores son realmente únicos?
if not ag["ID_Agencia"].is_unique:
    print("Ojo, hay IDs duplicados")

In [ ]:
# No existan duplicados ni valores nulos
ag['ID_Agencia'].is_unique and ag['ID_Agencia'].notna().all()

In [ ]:
# Columna nombre agencia
ag["Agencia_nombre"]

In [ ]:
col = ag['Agencia_nombre']
print("Total registros:", len(col))
print("Valores únicos:", col.nunique())
print("Nulos:", col.isna().sum())
print("Duplicados exactos:", col.duplicated().sum())

In [ ]:
ag['Agencia_nombre'].value_counts(dropna=False)

In [ ]:
# Para normalizar caracteres
import unicodedata
import pandas as pd

def normalizar_nombre(nombre):
    if pd.isna(nombre):
        return None

    nombre = str(nombre).strip().lower()

    nombre = " ".join(nombre.split())

    nombre = "".join(
        c for c in unicodedata.normalize("NFKD", nombre)
        if not unicodedata.combining(c)
    )

    return nombre

ag['agencia_normalizada'] = (
    ag['Agencia_nombre'].apply(normalizar_nombre)
)

In [ ]:
ag

In [ ]:
ag[
    ag['agencia_normalizada'].notna()
    & ag['agencia_normalizada'].duplicated(keep=False)
].sort_values('agencia_normalizada')

In [ ]:
from rapidfuzz import fuzz
from itertools import combinations

agencias = (
    ag['agencia_normalizada']
    .dropna()
    .unique()
)

resultados = []

for agencia1, agencia2 in combinations(agencias, 2):
    similitud = fuzz.ratio(agencia1, agencia2)

    if similitud >= 50:
        resultados.append({
            'agencia_1': agencia1,
            'agencia_2': agencia2,
            'similitud': round(similitud, 2)
        })

df_similares = pd.DataFrame(
    resultados,
    columns=['agencia_1', 'agencia_2', 'similitud']
).sort_values('similitud', ascending=False)



In [ ]:
from IPython.display import display

display(df_similares)

Ya que hay mucho ruido por extensiones de nombres que comparten el "sociedad anónima de capital variable" las eliminaré primero.

In [ ]:
import re
from collections import Counter

# 1. Extraer todas las palabras individuales de tus 133 registros
tokens = []
for nombre in ag['agencia_normalizada'].dropna().unique():
    palabras = re.findall(r'[a-záéíóúñ]+', nombre.lower())
    tokens.extend(palabras)

conteo = Counter(tokens)

# 2. Filtrar palabras que aparecen solo 1 vez (aquí se esconden los typos)
palabras_unicas = sorted([palabra for palabra, freq in conteo.items() if freq == 1])
print(f"Total de palabras únicas a revisar: {len(palabras_unicas)}")
print(palabras_unicas)

In [ ]:
import difflib

frecuentes = [p for p, f in conteo.items() if f >= 2]

for p in palabras_unicas:
    parecidas = difflib.get_close_matches(p, frecuentes, n=1, cutoff=0.8)
    if parecidas:
        print(f"{p:20} → ¿{parecidas[0]}? ({conteo[parecidas[0]]} apariciones)")

Voy a analizar las columnas del dataframe para empresas

In [ ]:
em = dfs["empresas"]

In [ ]:
em.info()

In [ ]:
# No existan duplicados ni valores nulos en su campo identificador
em['ID_empresa'].is_unique and em['ID_empresa'].notna().all()

In [ ]:
# Aprovechando que es un df de pocos valores - Valido valores únicos
em["Empresa_nombre"].value_counts()

In [ ]:
em["Empresa_nombre"].unique()

In [ ]:
# Explorando cantidad de habitaciones mín y máxima
mínimo = em["Habitaciones_tot"].min()
máximo = em["Habitaciones_tot"].max()
print ("Habitaciones min: " + str(mínimo) + " máxi: " + str(máximo))

In [ ]:
# en qué hotel están? (posición del df)
em["Habitaciones_tot"].idxmin()

In [ ]:
em["Habitaciones_tot"].idxmax()

In [ ]:
em.describe()

# Explorando ocupaciones

In [ ]:
oc = dfs["ocupaciones"]

In [ ]:
oc.info()

In [ ]:
oc

# Ahora reservaciones

In [ ]:
re = dfs["reservaciones"]

In [ ]:
re.info()

In [ ]:
re

# Ahora, segmentos

In [ ]:
sm = dfs["segmentos_mercados"]

In [ ]:
sm.info()

In [ ]:
sm

In [ ]:
oc = dfs["ocupaciones"]
ag_ids = set(dfs["agencias"]["ID_Agencia"])

catalogada = oc["ID_Agencia"].isin(ag_ids).rename("catalogada")
impacto = oc.groupby(catalogada).agg(
    filas=("ID_Agencia", "size"),
    agencias=("ID_Agencia", "nunique"),
    cuartos_noche=("cuartos_noche", "sum"),
    ingresos=("ingresos_habitaciones", "sum"),
)
(impacto / impacto.sum() * 100).round(2)

In [ ]:
dfs["agencias"]["ID_Agencia"].agg(["min", "max", "nunique"])
oc["ID_Agencia"].agg(["min", "max", "nunique"])